<a href="https://colab.research.google.com/github/jwe4/makemore_from_scratch/blob/main/build_makemore2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
#read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
len(words)

32033

In [4]:
# build the vocabulart of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [5]:
# build the dataset
block_size = 3 # context length: how any characters do we take to predict the next one
# X is input, Y is labels

X,Y = [],[]
for w in words[:5]:
  print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix= stoi[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(itos[i] for i in context), '-->', itos[ix])
    context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... --> e
..e --> m
.em --> m
emm --> a
mma --> .
olivia
... --> o
..o --> l
.ol --> i
oli --> v
liv --> i
ivi --> a
via --> .
ava
... --> a
..a --> v
.av --> a
ava --> .
isabella
... --> i
..i --> s
.is --> a
isa --> b
sab --> e
abe --> l
bel --> l
ell --> a
lla --> .
sophia
... --> s
..s --> o
.so --> p
sop --> h
oph --> i
phi --> a
hia --> .


In [6]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [7]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 19],
        [ 9, 19,  1],
        [19,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 12],
        [ 5, 12, 12],
        [12, 12,  1],
        [ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19, 15],
        [19, 15, 16],
        [15, 16,  8],
        [16,  8,  9],
        [ 8,  9,  1]])

In [8]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [9]:
# the 27 character embedding
C = torch.randn((27,2))

In [10]:
C

tensor([[ 1.0697, -0.5071],
        [-0.7834, -0.4647],
        [-0.2177,  1.9389],
        [ 0.8693,  0.3078],
        [-0.4593, -1.5778],
        [ 1.9072,  0.2075],
        [ 0.9233, -0.8432],
        [ 1.0479,  0.1682],
        [-2.0945,  1.2877],
        [-1.0239, -0.8959],
        [-0.8286,  0.9218],
        [ 0.9067,  0.4883],
        [ 1.5614, -0.2039],
        [ 1.0953, -2.0160],
        [-1.1814, -0.1689],
        [ 1.3611,  0.6911],
        [-0.7749,  0.1821],
        [ 1.3055,  0.6864],
        [ 1.1701,  0.3692],
        [-0.0793,  0.2907],
        [ 1.6326, -1.9275],
        [ 0.1581,  0.5086],
        [-1.0338, -0.0769],
        [-0.9755,  0.6985],
        [-1.4544, -0.9824],
        [-0.1998,  2.0557],
        [-0.5504, -1.0131]])

In [11]:
# encode an example integer( integers represent the characters)

F.one_hot(torch.tensor(5), num_classes=27)

tensor([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0])

In [12]:
F.one_hot(torch.tensor(5), num_classes=27).float() @ C

tensor([1.9072, 0.2075])

In [13]:
C[5]

tensor([1.9072, 0.2075])

In [14]:
# will use index rather than one hot
# can index by X
C[X].shape

torch.Size([32, 3, 2])

In [15]:
X[13,2]

tensor(1)

In [16]:
C[X][13,2]
# Yes, the weight matrix W1 for the hidden layer is defined as (6, 100), meaning 6 rows and 100 columns.

tensor([-0.7834, -0.4647])

In [17]:
C[1]

tensor([-0.7834, -0.4647])

In [18]:
# so have embedding
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [19]:
# now building the hidden layer
W1 = torch.randn(6,100)
b1 = torch.randn(100)

In [20]:
emb[:,0,:].shape

torch.Size([32, 2])

In [21]:
emb[:,0,:]

tensor([[ 1.0697, -0.5071],
        [ 1.0697, -0.5071],
        [ 1.0697, -0.5071],
        [ 1.9072,  0.2075],
        [ 1.0953, -2.0160],
        [ 1.0697, -0.5071],
        [ 1.0697, -0.5071],
        [ 1.0697, -0.5071],
        [ 1.3611,  0.6911],
        [ 1.5614, -0.2039],
        [-1.0239, -0.8959],
        [-1.0338, -0.0769],
        [ 1.0697, -0.5071],
        [ 1.0697, -0.5071],
        [ 1.0697, -0.5071],
        [-0.7834, -0.4647],
        [ 1.0697, -0.5071],
        [ 1.0697, -0.5071],
        [ 1.0697, -0.5071],
        [-1.0239, -0.8959],
        [-0.0793,  0.2907],
        [-0.7834, -0.4647],
        [-0.2177,  1.9389],
        [ 1.9072,  0.2075],
        [ 1.5614, -0.2039],
        [ 1.0697, -0.5071],
        [ 1.0697, -0.5071],
        [ 1.0697, -0.5071],
        [-0.0793,  0.2907],
        [ 1.3611,  0.6911],
        [-0.7749,  0.1821],
        [-2.0945,  1.2877]])

In [22]:
torch.cat([emb[:,0,:], emb[:,1,:],emb[:,2,:]],1).shape

torch.Size([32, 6])

In [23]:
# can do something similar with unbind
torch.cat(torch.unbind(emb,1),1).shape

torch.Size([32, 6])

In [24]:
# still another approach
a = torch.arange(18)
a

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])

In [25]:
a.shape

torch.Size([18])

In [26]:
a.view(9,2) # view is very efficient

tensor([[ 0,  1],
        [ 2,  3],
        [ 4,  5],
        [ 6,  7],
        [ 8,  9],
        [10, 11],
        [12, 13],
        [14, 15],
        [16, 17]])

In [27]:
emb.shape

torch.Size([32, 3, 2])

In [28]:
emb.view(32,6) == torch.cat(torch.unbind(emb,1),1)
#


tensor([[True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, T

In [29]:
# back to the hidden layer interface
W1 = torch.randn(6,100)
b1 = torch.randn(100)
h = emb.view(32,6) @ W1 + b1
h.shape

torch.Size([32, 100])

In [32]:
# h = emb.view(emb.shape[0], 6) @W1 + b1
# equivalently, pytorch will infer 32 since it knows the dimensions of W1 + b1
h = emb.view(-1,6) @ W1 + b1

In [33]:
h.shape

torch.Size([32, 100])

In [34]:
h = torch.tanh(emb.view(-1,6) @ W1 + b1) # to make probabilities

In [35]:
h

tensor([[ 0.9702,  0.9963, -0.9855,  ..., -0.0370,  0.9462,  0.9947],
        [ 0.9823,  0.9818, -0.9973,  ...,  0.8347,  0.7027,  0.9806],
        [ 0.9353,  1.0000, -0.9869,  ...,  0.2788,  0.9644,  0.9977],
        ...,
        [-0.9989, -0.5356, -0.1244,  ..., -0.9802, -0.9630,  0.9924],
        [-0.9757, -0.9891,  0.2843,  ...,  0.5285, -0.9798, -0.9997],
        [ 0.9976,  0.9998, -0.0763,  ...,  0.9995, -0.9864, -1.0000]])

In [36]:
h.shape

torch.Size([32, 100])

In [38]:
# with regard to W1 + b1
# have broadcasting
# 32, 100
#     100

# this results in align on right and creating a 1 on the left
# 1 1000 -- creates a 1 x 100 row that gets added as a column to each of the 32

W2 = torch.randn(100,27)
b2 = torch.randn(27)

logits = h @ W2 + b2


In [39]:
logits.shape

torch.Size([32, 27])

In [41]:
counts = logits.exp()


In [42]:
prob = counts / counts.sum(1,keepdims=True)

In [ ]:
prob.shape()

In [43]:
prob[0].sum()

tensor(1.)

In [44]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [46]:
# now to get the negative log likelihood loss by comparing with the probabilites in Y
prob[torch.arange(32),Y]


tensor([3.0344e-03, 3.3713e-05, 7.2534e-07, 6.1864e-19, 1.5184e-13, 5.1282e-11,
        3.7330e-12, 4.1709e-08, 1.6923e-05, 7.4719e-06, 4.1461e-05, 2.9751e-07,
        1.6163e-09, 2.9809e-08, 2.0122e-10, 6.2017e-07, 8.9117e-08, 1.1164e-05,
        2.3649e-07, 1.1505e-04, 4.8271e-03, 8.6576e-15, 1.6573e-17, 1.1372e-10,
        7.4674e-13, 1.4582e-02, 5.6039e-12, 6.9471e-01, 6.4793e-01, 7.3352e-06,
        1.7744e-03, 1.5999e-06])

In [47]:
loss = -prob[torch.arange(32),Y].log().mean()
loss

tensor(16.6359)

In [ ]:
# ---- now made respectab le :) ----

In [59]:
X.shape, Y.shape # dataset

(torch.Size([32, 3]), torch.Size([32]))

In [60]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27,2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g) # Added generator=g here
W2 = torch.randn((100,27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [61]:
sum(p.nelement() for p in parameters) # number of parameters in total

3481

In [62]:
emb = C[X] # (32,3,2)
h = torch.tanh(emb.view(-1,6) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2  # (32,27)
counts = logits.exp()
prob = counts / counts.sum(1, keepdims=True)
loss = -prob[torch.arange(32), Y].log().mean()
loss


tensor(17.7697)

In [63]:
F.cross_entropy(logits, Y) # can use builtin function for this, it is more efficient

tensor(17.7697)

In [64]:
emb = C[X] # (32,3,2)
h = torch.tanh(emb.view(-1,6) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2  # (32,27)
F.cross_entropy(logits, Y)
loss
# don't need to create the intermediate tensors, runs in a fused kernel

tensor(17.7697)

In [65]:
# can have problems like this in extreme cases
# caused by problems with floating point range
# pytorch calculates the greatest number and subtracts it
# doesn't affect the accuracy
logits = torch.tensor([-100,-3,0,100])
counts = logits.exp()
probs = counts/counts.sum()
probs

tensor([0., 0., 0., nan])

In [79]:
# build the dataset again for all the words
block_size = 3 # context length: how any characters do we take to predict the next one
# X is input, Y is labels

X,Y = [],[]
for w in words:
  # print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix= stoi[ch]
    X.append(context)
    Y.append(ix)
    # print(''.join(itos[i] for i in context), '-->', itos[ix])
    context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

In [80]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [81]:
# re-initialize the weights
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27,2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g) # Added generator=g here
W2 = torch.randn((100,27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [82]:
for p in parameters:
  p.requires_grad = True

In [83]:

for _ in range(10):
  #forward pass

  emb = C[X] # (32,3,2)
  h = torch.tanh(emb.view(-1,6) @ W1 + b1) # (32, 100)
  logits = h @ W2 + b2  # (32,27)
  loss = F.cross_entropy(logits, Y)
  print(loss.item())
  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()
  #update
  for p in parameters:
    p.data += -0.1 * p.grad

19.505226135253906
17.08449363708496
15.776531219482422
14.833340644836426
14.002603530883789
13.253260612487793
12.57991886138916
11.983101844787598
11.47049331665039
11.051856994628906


In [ ]:
torch.randint(0, X.shape[0], (32,))

In [84]:
torch.randint(0, X.shape[0], (32,))

tensor([ 56698,  16521,  33712, 164885, 177337,  87954,  98559, 150782, 126959,
         76423, 156680,  95275,  44916,  83001, 138535, 150527, 227206,  37227,
        111921,  54821, 215424, 152424,  65627, 103312, 226326,    241, 155785,
         59087, 146473, 227446,  32410, 126069])

In [90]:
for _ in range(1000):
  #forward pass
  # minibatch construct
  ix = torch.randint(0, X.shape[0], (32,))

  # only get a mini batch
  emb = C[X[ix]] # (32,3,2)
  h = torch.tanh(emb.view(-1,6) @ W1 + b1) # (32, 100)
  logits = h @ W2 + b2  # (32,27)
  loss = F.cross_entropy(logits, Y[ix])

  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()
  #update
  for p in parameters:
    p.data += -0.1 * p.grad

print(loss.item())

2.431828737258911


In [91]:
# now evaluate total loss after mini batches
emb = C[X] # (32,3,2)
h = torch.tanh(emb.view(-1,6) @ W1 + b1) # (32,100)
logits = h @ W2 + b2 # (32,27)
loss = F.cross_entropy(logits, Y)
loss

tensor(2.6003, grad_fn=<NllLossBackward0>)